# Heartbeat v2 Validation

Reads the latest object from `raw/heartbeat_v2/` and prints the payload.

In [ ]:
from minio import Minio
from datetime import timezone

client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

bucket_name = "raw"
prefix = "heartbeat_v2/"
objects = list(client.list_objects(bucket_name, prefix=prefix, recursive=True))

if not objects:
    raise RuntimeError("No heartbeat_v2 files found in raw bucket")

latest = max(objects, key=lambda o: o.last_modified)
print(f"Latest file: {latest.object_name}")
print(f"Last modified: {latest.last_modified.astimezone(timezone.utc)}")

response = client.get_object(bucket_name, latest.object_name)
try:
    content = response.read().decode("utf-8")
    print("\nFile contents:\n")
    print(content)
finally:
    response.close()
    response.release_conn()
